## Importation

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm 
import numpy as np
import os
import pandas as pd

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Moteur de calcul activé : {device.type.upper()}")

 Moteur de calcul activé : CUDA


In [3]:
mon_dataset=pd.read_csv(r"C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\processed\StockTwits_2020_2022_cleaned_final.csv")

In [4]:
mon_dataset

,Tweet,Ticker,Jour,Heure_decimale,Texte_Nettoye
0,$AAPL good things happening 2020 run trump and...,AAPL,2020-01-01,0.100000,$AAPL good things happening 2020 run trump and...
1,$AAPL Happy New Year amazing winning AAPL Bull...,AAPL,2020-01-01,0.133333,$AAPL Happy New Year amazing winning AAPL Bull...
2,Happy New Year :)\n$AAPL $TSLA $AMZN $SPY $B...,AAPL,2020-01-01,0.233333,Happy New Year :) $AAPL $TSLA $AMZN $SPY $BTC.X
3,@Taxes_R2_Damn_High my dad ended this year by...,AAPL,2020-01-01,0.316667,my dad ended this year by selling half his $aa...
4,$AAPL And to those using the tired and old del...,AAPL,2020-01-01,0.616667,$AAPL And to those using the tired and old del...
...,...,...,...,...,...
3711328,$TSLA 777,TSLA,2022-02-28,20.783333,$TSLA 777
3711329,$AMC $TSLA \n\nCan’t wait to buy a Tesla for o...,TSLA,2022-02-28,20.800000,$AMC $TSLA Can’t wait to buy a Tesla for overa...
3711330,$TSLA with future up now. any guess what will ...,TSLA,2022-02-28,20.800000,$TSLA with future up now. any guess what will ...
3711331,$TSLA what are the chances of Biden mentioning...,TSLA,2022-02-28,20.916667,$TSLA what are the chances of Biden mentioning...


In [ ]:
import os
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

nom_modele = "yiyanghkust/finbert-tone"
tokenizer = AutoTokenizer.from_pretrained(nom_modele)
model = AutoModelForSequenceClassification.from_pretrained(nom_modele, use_safetensors=True).to(device)
model.eval() # Désactive l'apprentissage pour accélérer

# Le fichier où sera stocké le résultat (comme un coffre-fort)
chemin_sauvegarde = r"C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\processed\02_StockTwits_SCORED_2020_2022.csv"

# 2. Système Anti-Coupure (Checkpointing)
lignes_deja_traitees = 0
if os.path.exists(chemin_sauvegarde):
    # On lit uniquement la première colonne du CSV existant pour compter rapidement les lignes
    df_existant = pd.read_csv(chemin_sauvegarde, usecols=[0])
    lignes_deja_traitees = len(df_existant)
    del df_existant # Libère la RAM
    print(f"🔄 Reprise automatique détectée : Démarrage à partir de la ligne {lignes_deja_traitees:,} / {len(mon_dataset):,}")
else:
    print(f" Lancement initial du traitement sur {len(mon_dataset):,} lignes...")

BATCH_SIZE = 128
CHUNK_SIZE = 5120  # On sauvegarde physiquement sur le disque dur toutes les 5120 lignes

# 3. La boucle d'Inférence Industrielle
with torch.no_grad(): # Économise 50% de mémoire graphique
    # On avance par "gros blocs" de 5120 lignes en partant d'où on s'est arrêté
    for chunk_start in tqdm(range(lignes_deja_traitees, len(mon_dataset), CHUNK_SIZE), desc="Progression Globale"):
        chunk_end = min(chunk_start + CHUNK_SIZE, len(mon_dataset))
        
        # On isole le bloc de données
        df_chunk = mon_dataset.iloc[chunk_start:chunk_end].copy()
        
        # CORRECTION ICI : Remplacement des NaN par du vide et conversion stricte en String
        textes_chunk = df_chunk['Texte_Nettoye'].fillna("").astype(str).tolist()
        
        scores_neutres, scores_positifs, scores_negatifs = [], [], []
        
        # On traite ce bloc par batchs de 128 pour la carte graphique
        for i in range(0, len(textes_chunk), BATCH_SIZE):
            batch_textes = textes_chunk[i:i + BATCH_SIZE]
            
            inputs = tokenizer(batch_textes, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
            outputs = model(**inputs)
            
            # Softmax pour obtenir les pourcentages
            probabilites = torch.nn.functional.softmax(outputs.logits, dim=-1).cpu().numpy()
            
            for prob in probabilites:
                scores_neutres.append(prob[0])
                scores_positifs.append(prob[1])
                scores_negatifs.append(prob[2])
        
        # On ajoute les scores au bloc actuel
        df_chunk['FinBERT_Positive'] = scores_positifs
        df_chunk['FinBERT_Negative'] = scores_negatifs
        df_chunk['FinBERT_Neutral'] = scores_neutres
        
        header_flag = not os.path.exists(chemin_sauvegarde)
        df_chunk.to_csv(chemin_sauvegarde, mode='a', header=header_flag, index=False)
        
        del df_chunk, textes_chunk, scores_neutres, scores_positifs, scores_negatifs

🔄 Reprise automatique détectée : Démarrage à partir de la ligne 3,640,320 / 3,711,333


Progression Globale: 100%|██████████| 14/14 [29:42<00:00, 127.31s/it]


In [6]:
df=pd.read_csv(chemin_sauvegarde)

In [7]:
df.head()

,Tweet,Ticker,Jour,Heure_decimale,Texte_Nettoye,FinBERT_Positive,FinBERT_Negative,FinBERT_Neutral
0,$AAPL good things happening 2020 run trump and...,AAPL,2020-01-01,0.100000,$AAPL good things happening 2020 run trump and...,0.200863,2.107586e-03,0.797029
1,$AAPL Happy New Year amazing winning AAPL Bull...,AAPL,2020-01-01,0.133333,$AAPL Happy New Year amazing winning AAPL Bull...,0.999996,4.646351e-07,0.000004
2,Happy New Year :)\n$AAPL $TSLA $AMZN $SPY $B...,AAPL,2020-01-01,0.233333,Happy New Year :) $AAPL $TSLA $AMZN $SPY $BTC.X,0.000071,1.083000e-05,0.999918
3,@Taxes_R2_Damn_High my dad ended this year by...,AAPL,2020-01-01,0.316667,my dad ended this year by selling half his $aa...,0.000117,9.996450e-01,0.000238
4,$AAPL And to those using the tired and old del...,AAPL,2020-01-01,0.616667,$AAPL And to those using the tired and old del...,0.002282,9.238381e-01,0.073880
